In [1]:
# 라이브러리 import
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

### 데이터 불러오기 및 확인

In [2]:
df = pd.read_excel('data/E Commerce Dataset.xlsx', sheet_name='E Comm')

df.head()

,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   str    
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   str    
 7   Gender                       5630 non-null   str    
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   str    
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   str    
 13  NumberOfAddress              

In [4]:
df.shape

(5630, 20)

In [5]:
# 중복 행 확인
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [6]:
print(df.columns.tolist())

['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']


In [7]:
# churn 컬럼 bool 타입으로 변환
df['Churn'] = df['Churn'].astype(bool)


# 컬럼을 데이터 타입별로 분류
float_columns = df.select_dtypes(include=['float']).columns
print()
print("Float columns are: " + ", ".join(float_columns))
print()

object_columns = df.select_dtypes(include=['object']).columns
print("Object columns are: " + ", ".join(object_columns))
print()

ordinal_columns = df[['CityTier','SatisfactionScore', 'Complain' ]].columns
print("Ordinal columns are: " + ", ".join(ordinal_columns))


Float columns are: Tenure, WarehouseToHome, HourSpendOnApp, OrderAmountHikeFromlastYear, CouponUsed, OrderCount, DaySinceLastOrder, CashbackAmount

Object columns are: PreferredLoginDevice, PreferredPaymentMode, Gender, PreferedOrderCat, MaritalStatus

Ordinal columns are: CityTier, SatisfactionScore, Complain


In [8]:
# 결측치 확인
missing_values = df.isnull().sum()
missing_columns = missing_values[missing_values > 0]
print(missing_columns) 

Tenure                         264
WarehouseToHome                251
HourSpendOnApp                 255
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
dtype: int64


In [9]:
# 결측치 비율 확인
missing_ratio = (df.isnull().sum() / len(df)) * 100
missing_ratio = missing_ratio[missing_ratio > 0].sort_values(ascending=False)

print("결측치 비율(%):")
print(missing_ratio)

결측치 비율(%):
DaySinceLastOrder              5.452931
OrderAmountHikeFromlastYear    4.706927
Tenure                         4.689165
OrderCount                     4.582593
CouponUsed                     4.547069
HourSpendOnApp                 4.529307
WarehouseToHome                4.458259
dtype: float64


- 수치형데이터에서 결측치가 너무 많음

In [10]:
# 범주형 값 확인
print("PreferredLoginDevice")
print(df['PreferredLoginDevice'].value_counts())
print()

print("PreferedOrderCat")
print(df['PreferedOrderCat'].value_counts())
print()

print("PreferredPaymentMode")
print(df['PreferredPaymentMode'].value_counts())

PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64


In [11]:
# 범주형 값 통일
df['PreferredLoginDevice'] = df['PreferredLoginDevice'].replace(
    ['Phone', 'Mobile Phone'], 'Mobile'
)

df['PreferedOrderCat'] = df['PreferedOrderCat'].replace(
    'Mobile Phone', 'Mobile'
)

df['PreferredPaymentMode'] = df['PreferredPaymentMode'].replace({
    'COD': 'Cash on Delivery',
    'CC': 'Credit Card'
})

print("PreferredLoginDevice")
print(df['PreferredLoginDevice'].value_counts())
print()

print("PreferedOrderCat")
print(df['PreferedOrderCat'].value_counts())
print()

print("PreferredPaymentMode")
print(df['PreferredPaymentMode'].value_counts())

PreferredLoginDevice
PreferredLoginDevice
Mobile      3996
Computer    1634
Name: count, dtype: int64

PreferedOrderCat
PreferedOrderCat
Mobile                2080
Laptop & Accessory    2050
Fashion                826
Grocery                410
Others                 264
Name: count, dtype: int64

PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1774
E wallet             614
Cash on Delivery     514
UPI                  414
Name: count, dtype: int64


In [12]:
print("전처리 후 데이터 크기:", df.shape)
print()
df.info()

전처리 후 데이터 크기: (5630, 20)

<class 'pandas.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   bool   
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   str    
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   str    
 7   Gender                       5630 non-null   str    
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   str    
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   str    
 13  Num

In [13]:
df.to_csv('data/churn_preprocessed.csv', index=False)
print("파일 저장 완료: churn_preprocessed.csv")

파일 저장 완료: churn_preprocessed.csv
